[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/06-agentic-ai/02-mbox_as_an_openai_function.ipynb)

# M|BOX as an OpenAI Function

Large language models are good at understanding what a user wants, and unreliable at recalling exact facts, a product name, a price, an inventory record. Function calling closes that gap: instead of asking a model to remember your data, you give it a real lookup tool backed by your actual index, and it answers using what comes back.

This notebook builds a complete OpenAI function on top of M|BOX: a product search that combines fuzzy name matching with a price constraint, describes itself to the API with a clear schema, and returns exactly the information a model needs to give an honest answer, including when a match is weak, or excluded only because of budget.

In this notebook you will:

1. Build an index with both text and numeric fields
2. Write a function that combines fuzzy name matching with a price constraint
3. Design that function's return values around every outcome a model needs to handle, not just success
4. Describe the function to the OpenAI API with a schema that tells the model how to use it
5. Let the model call the function, and ground its final answer in the result
6. Pick up practical habits for keeping this reliable in production

>Note: To run this notebook, you need to specify a `.env` file in your directory and specify your `OPENAI_API_KEY` inside. |


In [ ]:
# !pip install mbox openai

## 1. Build the index

A small product catalog, including `unit_price` as an indexed field. The function you are about to write needs to filter on price, not just display it, so it has to be part of the index, not left out.

In [1]:
import pandas as pd
from mbox.indexing import TableIndexer

df = pd.DataFrame({
    "product_id": ["B88-EXT", "A12-PWR", "C99-SNS", "D45-REL"],
    "product_name": ["Extended Battery Pack", "Portable Power Bank", "Motion Sensor Camera", "Smart Relay Switch"],
    "description": [
        "Rechargeable power cell for outdoor gear",
        "Compact portable power bank for phones and tablets",
        "Motion sensor camera with night vision for home security",
        "Smart relay switch for home automation systems"
    ],
    "unit_price": [24.99, 39.50, 59.00, 18.75]
})

index = TableIndexer.create_index(
    df,
    index_columns=["product_name", "description", "unit_price"],
    tmp_dir="tmp_index"
)

df

mbpie: 33 modules, 566 methods, 8 classes, 18 enums
  args: 426 required, 254 optional, 37 keywords, 39 flags, 26 arrays
  types: 372 int, 337 str, 1 double, 72 object


,product_id,product_name,description,unit_price
0,B88-EXT,Extended Battery Pack,Rechargeable power cell for outdoor gear,24.99
1,A12-PWR,Portable Power Bank,Compact portable power bank for phones and tab...,39.50
2,C99-SNS,Motion Sensor Camera,Motion sensor camera with night vision for hom...,59.00
3,D45-REL,Smart Relay Switch,Smart relay switch for home automation systems,18.75


## 2. Write the search function

A tool a model can call needs two things a plain dictionary lookup does not naturally give you: fuzzy tolerance for how the user actually types, and the ability to combine a text search with a numeric constraint in one query. `search_products` below does both, matching `product_name` with `APPROX` and, when a `max_price` is given, filtering `unit_price` with `NUM_LOWER` at the same time.

There is one more design decision worth calling out directly. If a price constraint excludes every candidate, the function does not just report failure. It runs a second, unconstrained search to check whether a matching product exists at all. This matters because a model can only be as honest as the information you give it: without this second check, there is no way for the function to ever distinguish "nothing like this exists" from "it exists, but costs more than the user asked for", the two are completely different answers, and only one of them is currently discoverable by the tool unless you build this in.

In [2]:
from mbox.recall import TableRecallConfig, TableRecallFieldConfig, TableRecallMode

def _text_only_match(product_name, max_results=3):
    config = TableRecallConfig(
        fields=[TableRecallFieldConfig(input_column="product_name", indexed_column="product_name",
                                        minimum_quality=0, weight=100, mode=TableRecallMode.APPROX)],
        max_results=max_results, min_total_match_value=0, include_field_scores=True
    )
    return index.match(queries=pd.DataFrame({"product_name": [product_name]}), config=config)

def search_products(product_name: str, max_price: float = None, max_results: int = 3) -> dict:
    """Search the product catalog by name, optionally constrained by a maximum price."""
    if max_price is None:
        r = _text_only_match(product_name, max_results)
        if len(r) == 0 or r["index_row"].iloc[0] == -1:
            return {"found": False, "matches": []}
        matches = [{"product_name": row["product_name_candidate"],
                    "match_confidence": int(row["product_name_score"]),
                    "overall_score": int(row["overall_score"])} for _, row in r.iterrows()]
        return {"found": True, "matches": matches}

    fields = [
        TableRecallFieldConfig(input_column="product_name", indexed_column="product_name",
                                minimum_quality=0, weight=70, mode=TableRecallMode.APPROX),
        TableRecallFieldConfig(input_column="unit_price", indexed_column="unit_price",
                                minimum_quality=0, weight=30, mode=TableRecallMode.NUM_LOWER)
    ]
    config = TableRecallConfig(fields=fields, max_results=max_results, min_total_match_value=0, include_field_scores=True)
    r = index.match(queries=pd.DataFrame({"product_name": [product_name], "unit_price": [max_price]}), config=config)

    if len(r) > 0 and r["index_row"].iloc[0] != -1:
        matches = [{"product_name": row["product_name_candidate"],
                    "match_confidence": int(row["product_name_score"]),
                    "overall_score": int(row["overall_score"])} for _, row in r.iterrows()]
        return {"found": True, "matches": matches}

    # Nothing satisfied the price constraint. Check whether a match exists WITHOUT it,
    # so the caller can tell "nothing like this exists" apart from "it exists, but costs more."
    unconstrained = _text_only_match(product_name, max_results=1)
    if len(unconstrained) > 0 and unconstrained["index_row"].iloc[0] != -1:
        row = unconstrained.iloc[0]
        actual_price = df.loc[df["product_name"] == row["product_name_candidate"], "unit_price"].iloc[0]
        return {
            "found": False,
            "matches": [],
            "note": "A matching product exists but exceeds the requested max_price.",
            "closest_match": {
                "product_name": row["product_name_candidate"],
                "match_confidence": int(row["product_name_score"]),
                "actual_price": float(actual_price)
            }
        }

    return {"found": False, "matches": []}

## 3. What the function returns, in every situation

A tool is only as useful as the range of answers it can give. Here is `search_products` handling three genuinely different situations a model will encounter constantly in real use.

In [3]:
import json

print("A confident match, no price constraint:")
print(json.dumps(search_products("extendd batery pak"), indent=2))

print("\nA matching product exists, but a price constraint excludes it:")
print(json.dumps(search_products("extendd batery pak", max_price=20), indent=2))

print("\nA genuinely weak, low-confidence match:")
print(json.dumps(search_products("Extendo Batory Pak"), indent=2))

A confident match, no price constraint:
{
  "found": true,
  "matches": [
    {
      "product_name": "Extended Battery Pack",
      "match_confidence": 60,
      "overall_score": 60
    }
  ]
}

A matching product exists, but a price constraint excludes it:
{
  "found": false,
  "matches": [],
  "note": "A matching product exists but exceeds the requested max_price.",
  "closest_match": {
    "product_name": "Extended Battery Pack",
    "match_confidence": 60,
    "actual_price": 24.99
  }
}

A genuinely weak, low-confidence match:
{
  "found": true,
  "matches": [
    {
      "product_name": "Extended Battery Pack",
      "match_confidence": 35,
      "overall_score": 35
    }
  ]
}


Three distinct signals, each meaning something different to whoever, or whatever, reads the response:

- A **confident match** resolves cleanly to `"Extended Battery Pack"` with `match_confidence: 60`, despite two typos in the query.
- A **budget-excluded match** returns `found: false`, but with a `closest_match` showing the real product and its real price, `$24.99`. This is not a failed search, it is a specific, useful answer: the thing exists, it just does not fit the constraint.
- A **weak match** still resolves to `"Extended Battery Pack"`, but with `match_confidence: 35`, low enough that it should be treated as a guess worth confirming, not a settled fact.

A model, or a person, reading these three responses has enough information to respond appropriately to each, which is the entire point of designing the function this way.

## 4. Describe the function to the OpenAI API

OpenAI's function calling expects a JSON schema describing the function's name, purpose, and parameters. The model uses this description to decide when the function is relevant and how to fill in its arguments, it never sees your Python code directly, so the description has to do real work, not just name the parameters.

Pay particular attention to how the description tells the model what to do with `match_confidence`, `found: false`, and `closest_match`. A model has no independent understanding of what a score of `35` means, or what an empty `matches` list combined with a populated `closest_match` implies, unless the function description spells it out.

In [4]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_products",
            "description": (
                "Search the product catalog by name, optionally filtering by a maximum price. "
                "Tolerates typos and partial matches. Always use this instead of guessing a "
                "product name from memory. Each match includes a match_confidence from 0 to 100; "
                "treat anything below 50 as uncertain and say so explicitly rather than presenting "
                "it as a confirmed match. If found is false but closest_match is present, a matching "
                "product exists but was excluded by max_price, tell the user it exists and mention "
                "its actual price, rather than saying nothing was found. If found is false and there "
                "is no closest_match, tell the user nothing matched at all, do not invent a "
                "plausible-sounding product."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "product_name": {
                        "type": "string",
                        "description": "The product name or description to search for, as typed by the user."
                    },
                    "max_price": {
                        "type": "number",
                        "description": "Optional maximum price the user is willing to pay."
                    },
                    "max_results": {
                        "type": "integer",
                        "description": "Maximum number of candidate matches to return.",
                        "default": 3
                    }
                },
                "required": ["product_name"]
            }
        }
    }
]

## 5. Call the model

Send a user message that plausibly needs both parameters, a fuzzy product name and a budget, so the model has a reason to fill in `max_price`, not just `product_name`.

In [5]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI()  

messages = [
    {
        "role": "system",
        "content": (
            "You help customers find products. Always look up product names using the search "
            "tool rather than guessing. Be honest about match confidence and about constraints "
            "like price that excluded a result."
        )
    },
    {"role": "user", "content": "Do you have anything like an extendd batery pak for under $20?"}
]

response = client.chat.completions.create(
    model="gpt-4o",
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

tool_calls = response.choices[0].message.tool_calls
print(tool_calls)

[ChatCompletionMessageFunctionToolCall(id='call_kNdGd9dP8vQjMJAgnvpQuUL6', function=Function(arguments='{"product_name":"extended battery pack","max_price":20}', name='search_products'), type='function')]


If the model parsed the request correctly, `tool_calls` should show `search_products` being called with both `product_name="extendd batery pak"` and `max_price=20`. Recognizing "under $20" as the `max_price` parameter, rather than just background text, depends directly on how clearly that parameter was described in Section 4.

## 6. Run the function, and send the result back

Parse the arguments the model requested, run the real function, and send the result back as a `tool` message. The query above should land on the budget-excluded case from Section 3.

In [6]:
tool_call = tool_calls[0]
arguments = json.loads(tool_call.function.arguments)

function_result = search_products(**arguments)

messages.append(response.choices[0].message)
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": json.dumps(function_result)
})

final_response = client.chat.completions.create(
    model="gpt-4o",
    messages=messages,
    tools=tools
)

print(final_response.choices[0].message.content)

I found an "Extended Battery Pack" that closely matches your request, but it costs $24.99, which is above your $20 limit. If you're interested or can adjust your budget, let me know! Unfortunately, there are no options available under $20.


A well-grounded answer here tells the user that an extended battery pack exists, at $24.99, above their $20 limit, rather than either inventing a cheaper product or simply saying nothing was found. That distinction is only possible because `search_products` returns `closest_match` explicitly when a price constraint is the sole reason for the empty result. Grounding is only ever as good as the information the tool actually provides, a model cannot infer facts a function's response does not contain.

## 7. Practical notes for production use

**Design your return values around every outcome, not just success.** A confident match, a weak match, and a match excluded by a constraint are three different situations. If your function's response cannot distinguish between them, no amount of prompt engineering will let the model tell them apart either.

**Keep the payload small.** Every field you include costs tokens on every subsequent turn. Return `product_name` and confidence scores; leave out anything the model does not need to answer the user, like internal `product_id` values, unless the conversation specifically requires them.

**Write the function description around all the outcomes it can produce.** Section 4's description explicitly tells the model what a low `match_confidence` means, and what `found: false` combined with `closest_match` means. Without that, the model is left guessing how much trust to place in a response it cannot fully interpret on its own.

**Test with deliberately messy and constrained input.** Test your function the way Section 3 did: a clean case, a constraint that excludes an otherwise good match, and a genuinely weak match, not just the tidy product names from your own catalog.

**Remember `tool_choice="auto"` means the model might not call the function at all.** If a user asks something unrelated to your catalog, the model may answer directly without calling `search_products`. This is usually correct, but if a lookup must always happen, `tool_choice` can be set to force a specific function instead.

## Next steps

- **`03-mbox_as_an_mcp_server.ipynb`** - expose M|BOX once, as a standard MCP server, instead of writing this integration separately for every model provider
- **`04-grounding_rag_with_deterministic_matching.ipynb`** - use this same grounding principle for retrieval-augmented generation

*M|BOX is currently in `beta`. Breaking changes may occur in minor releases until version `1.0.0`.*